[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/try/)

# anywidget-instruments gallery

Every widget in **control** and **indicator** mode (DOC-002). Run the cells, then interact with the controls: the indicators on the right follow through kernel callbacks.

**References:** IEC 60073 (lamp colors). These standards inspired the
widgets; the library does not claim conformity with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

> Running in JupyterLite (Pyodide): no threads, so stale-data detection is off and the loop below runs in the main thread.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import numpy as np

import anywidget_instruments as ai

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)

## Numeric controls → indicators

In [ ]:
knob = ai.Knob(30, step=1, unit="%", label="Knob (control)")
dial = ai.Dial(40, turns=3, label="Dial, 3 turns (control)")
slide = ai.FillSlide(55, label="FillSlide (control)")

gauge = ai.Gauge(
    30,
    unit="%",
    label="Gauge",
    peak_hold=True,
    ranges=[
        {"from": 0, "to": 60, "color": "#86efac"},
        {"from": 60, "to": 85, "color": "#fde047"},
        {"from": 85, "to": 100, "color": "#fca5a5"},
    ],
)
meter = ai.Meter(40, label="Meter")
tank = ai.Tank(55, unit="m³", label="Tank", markers=[20, 80], lo=15, hi=85, show_limits=True)

knob.on_change(lambda c: setattr(gauge, "value", c["new"]))
dial.on_change(lambda c: setattr(meter, "value", c["new"]))
slide.on_change(lambda c: setattr(tank, "value", c["new"]))

ai.Panel([knob, dial, slide, gauge, meter, tank], columns=3)

## More numeric displays

In [ ]:
thermo = ai.Thermometer(21.5, label="Thermometer")
vu = ai.VUMeter(72, label="VU", peak_hold=True)
seg = ai.SevenSegment(-12.7, digits=5, decimals=1, label="SevenSegment")
compass = ai.Compass(237, label="Compass")
log = ai.FillSlide(100, min=1, max=10_000, scale="log", format="%.3s", unit="Hz", label="Log scale")
semi = ai.Gauge(
    91,
    variant="semicircular",
    lo=10,
    hi=80,
    hihi=90,
    show_limits=True,
    label="Semicircular + alarms",
)
ai.Panel([thermo, vu, seg, compass, log, semi], columns=3)

In [ ]:
# indicator mode: knobs can be read-only too; disabled widgets reject input
ai.Panel(
    [
        ai.Knob(20, mode="indicator", label="Knob (indicator)"),
        ai.Knob(20, disabled=True, label="Disabled"),
        ai.Knob(float("nan"), label="NaN value"),
    ]
)

## Boolean controls and mechanical actions

In [ ]:
led = ai.LED(label="LED")
blink = ai.LED(label="Blinking LED", on_color="#ef4444", blink=True)
toggle = ai.ToggleSwitch(label="ToggleSwitch")
rocker = ai.RockerSwitch(label="RockerSwitch")
slide_sw = ai.SlideSwitch(label="SlideSwitch")
start = ai.PushButton(text="START", label="latch_when_released")
estop = ai.EmergencyStop(label="EmergencyStop")

toggle.on_change(lambda c: setattr(led, "value", c["new"]))
rocker.on_change(lambda c: setattr(blink, "value", c["new"]))

ai.Panel([led, blink, toggle, rocker, slide_sw, start, estop], columns=4)

In [ ]:
# a latched button is consumed by read_latched()
print("start pressed since last read:", start.read_latched())
print("emergency stop:", estop.value)

In [ ]:
estop.reset()

## Real-time waveform chart

In [ ]:
chart = ai.WaveformChart(
    n_traces=2,
    history=500,
    dt=0.01,
    x_unit="s",
    unit="V",
    autoscale_y=True,
    traces=[{"name": "sine"}, {"name": "noisy cosine"}],
    label="WaveformChart",
)
chart

In [ ]:
import time

t0 = 0
for _ in range(200):
    t = (t0 + np.arange(10)) * 0.01
    chart.append(
        np.column_stack(
            [np.sin(2 * np.pi * t), np.cos(2 * np.pi * 0.5 * t) + 0.1 * np.random.randn(10)]
        )
    )
    t0 += 10
    time.sleep(0.02)

In [ ]:
chart.update_mode = "sweep"  # try 'scope' and 'strip' too, and `chart.paused = True` to zoom

## Alarms

In [ ]:
alarm = ai.AlarmIndicator(
    alarm_id="TK-101.HI", priority="high", message="Tank level high", label="AlarmIndicator"
)
alarm.on_acknowledge(lambda e: print("acknowledged", e["alarm_id"]))
alarm.activate()
alarm

In [ ]:
alarm.clear()  # returns to normal once acknowledged

## Styles

In [ ]:
ai.Panel([ai.Gauge(62, style=s, label=s) for s in ai.STYLES], columns=3)